In [2]:
pip install pypdf langchain-community

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -U langchain langchain-openai langchain-community langchain-text-splitters langgraph faiss-cpu pypdf

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
    --------------------------------------- 0.3/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.3 MB 1.1 MB/s eta 0:00:15
   - -------------------------------------- 0.8/16.3 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.8/16.3 MB 1.1 MB/s eta 0:00:14
   -- ------------------------------------- 1.0/16.3 MB 1.1 MB/s eta 0:00:14
   --- ------------------------------------ 1.6/16.3 MB 1.2 MB/s eta 0:00:13
   --- ------------------------------------ 1.6/16.3 MB 1.2 MB/s eta 0:00:13
   ---- ----------------------------------- 1.8/16.3 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.1/16.3 MB 1.1 MB/s eta 0:00:13
   ----- ---------------------------------- 2.4/16.3 MB 1.1 MB/s eta 0:00:13
   ------ --------------------------------- 2.6/16.3 MB 1.2 MB/s eta 0:00:12
   ------- -------------------------------- 3.1/16.3 MB 1.2 MB/s eta 0:00:11
   -------- -

In [1]:
import os
from typing import TypedDict
#Defines the structure and expected data types of dictionary-based objects.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#Converts text into numerical vectors (embeddings) for semantic search.
from langchain_community.document_loaders import PyPDFLoader
#Loads and extracts text from PDF documents.
from langchain_community.vectorstores import FAISS
#Stores and searches vector embeddings efficiently for similarity-based retrieval.
from langchain_text_splitters import RecursiveCharacterTextSplitter
#Splits large text documents into samller schunks
from langchain_core.prompts import ChatPromptTemplate
#Creates structured prompts that provide instructions and input to an AI model.
from langgraph.graph import StateGraph, START, END
#Builds a workflow in LangGraph by connecting different processing nodes.

# 1. Read API Key

file_path = r"C:\LLM_Project_2026\LLM_Project_keys.txt"

with open(file_path, "r") as f:
    api_key = f.read().strip()

os.environ["OPENAI_API_KEY"] = api_key

print("API key loaded successfully!")

# 2. Initialize LLM and Embeddings

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)


embeddings = OpenAIEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

print("LLM initialized successfully")


# 3. Load Job Description PDF

pdf_path = r"C:\LLM_Project_2026\JD.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Number of JD pages loaded:", len(documents))


# 4. Split Documents into Chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks generated:", len(chunks))


# 5. Create Vector Database using FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("FAISS vector database created successfully")


# 6. Load Candidate Resume PDF

resume_path = r"C:\LLM_Project_2026\Student_profile1.pdf"

resume_loader = PyPDFLoader(resume_path)
resume_documents = resume_loader.load()

resume_text = "\n".join(
    doc.page_content for doc in resume_documents
)

print("Resume loaded successfully")


# 7. Define LangGraph State

class SkillGapState(TypedDict):
    resume: str
    retrieved_jds: str
    analysis: str


# 8. Define Retrieval Node

def retrieve_jds(state: SkillGapState):

    retrieved_docs = retriever.invoke(state["resume"])

    jd_context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    return {
        "retrieved_jds": jd_context
    }


# 9. Define Skill Gap Analysis Node

def analyze_skill_gap(state: SkillGapState):

    prompt = ChatPromptTemplate.from_template("""
    You are an expert HR and Technical Recruitment Analyst.

    Analyze the candidate resume against the retrieved
    Job Descriptions.

    CANDIDATE RESUME:
    {resume}

    RETRIEVED JOB DESCRIPTIONS:
    {retrieved_jds}

    Perform the following Skill Gap Analysis:

    1. Identify candidate technical skills.
    2. Identify candidate soft skills.
    3. Identify required skills from job descriptions.
    4. Identify matching skills.
    5. Identify missing technical skills.
    6. Identify missing soft skills.
    7. Calculate approximate skill match percentage.
    8. Identify suitable job roles.
    9. Suggest skills the candidate should learn.
    10. Provide recommendations to improve employability.

    Present the result in a clear structured format.

    Do not invent candidate qualifications or skills.
    """)

    chain = prompt | llm

    response = chain.invoke({
        "resume": state["resume"],
        "retrieved_jds": state["retrieved_jds"]
    })

    return {
        "analysis": response.content
    }


# 10. Create LangGraph Workflow

graph = StateGraph(SkillGapState)

graph.add_node("retrieve_jds", retrieve_jds)

graph.add_node("analyze_skill_gap", analyze_skill_gap)

graph.add_edge(START, "retrieve_jds")

graph.add_edge("retrieve_jds", "analyze_skill_gap")

graph.add_edge("analyze_skill_gap", END)


# 11. Compile LangGraph

skill_gap_graph = graph.compile()

print("LangGraph workflow compiled successfully")


# 12. Execute RAG Skill Gap Analysis

result = skill_gap_graph.invoke({
    "resume": resume_text,
    "retrieved_jds": "",
    "analysis": ""
})


# 13. Display Final Skill Gap Analysis

print("\n===================================")
print("       SKILL GAP ANALYSIS")
print("===================================")

print(result["analysis"])

C:\Users\HP\AppData\Local\Temp\ipykernel_1604\3207907339.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


API key loaded successfully!
LLM initialized successfully
Number of JD pages loaded: 7
Total chunks generated: 37
FAISS vector database created successfully
Resume loaded successfully
LangGraph workflow compiled successfully

       SKILL GAP ANALYSIS
### Skill Gap Analysis

#### 1. Candidate Technical Skills
- Java
- Full Stack Development
- JavaScript
- Bootstrap
- Frontend Development
- HTML/CSS

#### 2. Candidate Soft Skills
- Motivated
- Detail-oriented
- Eager to learn and contribute

#### 3. Required Skills from Job Descriptions
**Data Science Engineer**:
- SQL
- Python/Scala
- Apache Spark
- Airflow
- Data Warehouses (Snowflake, BigQuery, Redshift)
- Hadoop/Kafka
- Data Modeling
- Machine Learning / Statistical Modeling

**Data Scientist**:
- Cleaning and analyzing unstructured data
- Predictive modeling
- Machine Learning Algorithms

**Data Analyst**:
- SQL
- Microsoft Excel (Advanced)
- PowerBI / Tableau
- Basic Python/R
- Business Acumen
- Data Storytelling

**Full Stack Dev